<a href="https://colab.research.google.com/github/h124577999-art/dsd/blob/main/sdsdsdstitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Circle
import base64
import tempfile

# --- 網頁頁面配置 ---
st.set_page_config(page_title="戰術防禦模擬系統", layout="wide")

st.title("🛡️ 戰術防禦模擬：SUT 動態機動 vs 傳統定點佈署")
st.markdown("""
本系統模擬 100 次接戰過程，對比 **SUT (Small Unit Tactics)** 與 **傳統守衛** 在資產保護力與自身生存率上的差異。
所有統計圖表顯示每 10 次模擬的平均值。
""")

# --- 側邊欄：參數設定 ---
st.sidebar.header("🕹️ 戰術參數調校")

with st.sidebar.expander("敵軍進攻設定", expanded=True):
    e_count = st.slider("敵軍波次數量", 5, 25, 12)
    e_speed = st.slider("敵軍進攻速度", 0.5, 2.0, 1.15)

with st.sidebar.expander("防禦方性能", expanded=True):
    sut_hit_p = st.slider("SUT 基礎攔截率", 0.5, 1.0, 0.88)
    trad_hit_p = st.slider("傳統組基礎攔截率", 0.1, 0.8, 0.42)
    sut_dist_limit = st.slider("SUT 偵測半徑", 10, 30, 18)
    trad_dist_limit = st.slider("傳統組偵測半徑", 5, 20, 12)

# --- 核心模擬引擎 ---
def run_simulation(mode, e_num, e_spd, hit_p_base, d_limit):
    np.random.seed(42) # 固定種子確保單次執行結果一致
    results = []
    for _ in range(100):
        e_pos = np.random.rand(e_num, 2) * 45 + 5
        e_targets = np.array([110, 60]) + np.random.uniform(-30, 30, (e_num, 2))
        health, deaths = 100.0, 0

        if mode == "sut":
            units = [{'pos': np.array([60.0, 45.0]), 'active': True}, {'pos': np.array([60.0, 75.0]), 'active': True}]
            risk = 0.045
        else:
            units = [{'pos': np.array(p), 'active': True} for p in [[25, 60], [95, 60], [105, 35], [105, 85]]]
            risk = 0.012

        for _ in range(160):
            if len(e_pos) == 0 or health <= 0: break
            active_e_idx = []
            for u in units:
                if not u['active']: continue
                if mode == "sut":
                    dist_to_e = np.linalg.norm(e_pos - u['pos'], axis=1)
                    t_idx = np.argmin(dist_to_e)
                    u['pos'] += (e_pos[t_idx] - u['pos']) / 8.0
                for i in range(len(e_pos)):
                    if np.linalg.norm(u['pos'] - e_pos[i]) < 8 and np.random.rand() < risk:
                        u['active'] = False; deaths += 1; break
            for i in range(len(e_pos)):
                dir_v = e_targets[i] - e_pos[i]
                e_pos[i] += (dir_v / np.linalg.norm(dir_v)) * e_spd
                d_hvt = np.linalg.norm(e_pos[i] - np.array([110, 60]))
                if d_hvt < 8: health -= 3.5
                elif d_hvt < 20: health -= 0.5
                is_hit = False
                for u in units:
                    if u['active'] and np.linalg.norm(u['pos'] - e_pos[i]) < d_limit:
                        p_bonus = 1.0 - (np.linalg.norm(u['pos'] - e_pos[i]) / d_limit)
                        if np.random.rand() < hit_p_base * (0.7 + 0.3 * p_bonus):
                            is_hit = True; break
                if not is_hit: active_e_idx.append(i)
            e_pos = e_pos[active_e_idx]; e_targets = e_targets[active_e_idx]
        results.append([max(0, health), (deaths / len(units)) * 100])
    return np.array(results)

# --- 執行按鈕 ---
if st.sidebar.button("🚀 啟動模擬運算"):
    # 執行數據計算
    data_trad = run_simulation("trad", e_count, e_speed, trad_hit_p, trad_dist_limit)
    data_sut = run_simulation("sut", e_count, e_speed, sut_hit_p, sut_dist_limit)

    # 1. 顯示統計圖表
    st.subheader("📊 100 次接戰統計 (每 10 次平均值)")
    fig_stat, axs = plt.subplots(2, 2, figsize=(12, 10))
    group_indices = np.arange(1, 11)

    def plot_bars(ax, raw_data, col, title, color, ylabel):
        group_avgs = np.mean(raw_data[:, col].reshape(10, 10), axis=1)
        bars = ax.bar(group_indices, group_avgs, color=color, alpha=0.8, width=0.7, edgecolor='black', linewidth=1.2)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}', ha='center', va='bottom', fontweight='bold')
        ax.set_title(title, fontweight='bold'); ax.set_xticks(group_indices)
        ax.set_ylabel(ylabel); ax.set_ylim(0, 115); ax.grid(True, axis='y', alpha=0.3)

    plot_bars(axs[0, 0], data_trad, 0, "Trad: Health Avg", "red", "Health %")
    plot_bars(axs[0, 1], data_sut, 0, "SUT: Health Avg", "blue", "Health %")
    plot_bars(axs[1, 0], data_trad, 1, "Trad: Casualty Avg", "orange", "Rate %")
    plot_bars(axs[1, 1], data_sut, 1, "SUT: Casualty Avg", "cyan", "Rate %")
    st.pyplot(fig_stat)

    # 2. 生成接戰動畫
    st.subheader("🎬 戰術接戰動態演示 (無數值純淨模式)")
    fig_ani, (v1, v2) = plt.subplots(1, 2, figsize=(12, 6))
    TARGET = np.array([110, 60])

    def setup_map(ax, title):
        ax.set_xlim(0, 140); ax.set_ylim(0, 120); ax.set_facecolor('#fdfdfd')
        ax.add_patch(Rectangle((20, 30), 50, 60, color='gray', alpha=0.05))
        ax.add_patch(Circle(TARGET, 8, color='purple', alpha=0.15))
        ax.set_title(title, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([]) # 移除外框刻度

    setup_map(v1, "Traditional: Static"); setup_map(v2, "SUT: Dynamic")
    se1 = v1.scatter([], [], c='red', marker='x', s=60)
    sg1 = v1.scatter([], [], c='blue', marker='o', s=100)
    se2 = v2.scatter([], [], c='red', marker='x', s=60)
    sg2 = v2.scatter([], [], c='blue', marker='^', s=150)

    # 動態變數初始化
    ve1 = np.random.rand(e_count, 2) * 45 + 5; ve2 = ve1.copy()
    vg1 = [[25, 60, True], [95, 60, True], [105, 35, True], [105, 85, True]]
    vg2 = [[60, 45, True], [60, 75, True]]
    vh1, vh2 = [100.0], [100.0]

    def update_ani(f):
        global ve1, ve2
        # 動畫簡化更新邏輯 (與模擬引擎同步)
        for i, (v_pos, v_guards, v_hp, d_lim, h_p, m_idx) in enumerate([(ve1, vg1, vh1, trad_dist_limit, trad_hit_p, 1), (ve2, vg2, vh2, sut_dist_limit, sut_hit_p, 2)]):
            if v_hp[0] <= 0 or len(v_pos) == 0: continue
            if m_idx == 2:
                for g in v_guards:
                    if g[2] and len(v_pos) > 0:
                        idx = np.argmin(np.linalg.norm(v_pos - g[:2], axis=1))
                        g[:2] += (v_pos[idx] - g[:2]) / 8.0
            rem = []
            for j in range(len(v_pos)):
                v_pos[j] += (TARGET - v_pos[j]) / np.linalg.norm(TARGET - v_pos[j]) * e_speed
                if np.linalg.norm(v_pos[j] - TARGET) < 8: v_hp[0] -= 2.2
                hit = any(g[2] and np.linalg.norm(np.array(g[:2]) - v_pos[j]) < d_lim and np.random.rand() < h_p for g in v_guards)
                if not hit: rem.append(j)
            if m_idx == 1: globals()['ve1'] = v_pos[rem]
            else: globals()['ve2'] = v_pos[rem]

        se1.set_offsets(ve1 if len(ve1)>0 else np.empty((0,2))); sg1.set_offsets([g[:2] for g in vg1 if g[2]])
        se2.set_offsets(ve2 if len(ve2)>0 else np.empty((0,2))); sg2.set_offsets([g[:2] for g in vg2 if g[2]])
        return se1, sg1, se2, sg2

    ani = FuncAnimation(fig_ani, update_ani, frames=100, interval=50, blit=True)

    # 匯出動畫為 HTML/GIF
    with tempfile.NamedTemporaryFile(delete=False, suffix='.gif') as tmpfile:
        ani.save(tmpfile.name, writer='pillow', fps=20)
        file_ = open(tmpfile.name, "rb")
        contents = file_.read()
        data_url = base64.b64encode(contents).decode("utf-8")
        file_.close()
        st.markdown(f'<img src="data:image/gif;base64,{data_url}" width="100%">', unsafe_allow_html=True)
else:
    st.info("👈 請在左側調整參數並點擊『啟動模擬運算』。")